<a href="https://www.kaggle.com/code/aabdollahii/test-on-fabertwikifarsi-making-testset-sec6?scriptVersionId=335344387" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [15]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/notebooks/aabdollahii/finetuning-on-wikifarsi/__results__.html
/kaggle/input/notebooks/aabdollahii/finetuning-on-wikifarsi/__huggingface_repos__.json
/kaggle/input/notebooks/aabdollahii/finetuning-on-wikifarsi/__notebook__.ipynb
/kaggle/input/notebooks/aabdollahii/finetuning-on-wikifarsi/__output__.json
/kaggle/input/notebooks/aabdollahii/finetuning-on-wikifarsi/custom.css
/kaggle/input/notebooks/aabdollahii/finetuning-on-wikifarsi/fabert_kg_mlm/config.json
/kaggle/input/notebooks/aabdollahii/finetuning-on-wikifarsi/fabert_kg_mlm/training_args.bin
/kaggle/input/notebooks/aabdollahii/finetuning-on-wikifarsi/fabert_kg_mlm/tokenizer.json
/kaggle/input/notebooks/aabdollahii/finetuning-on-wikifarsi/fabert_kg_mlm/tokenizer_config.json
/kaggle/input/notebooks/aabdollahii/finetuning-on-wikifarsi/fabert_kg_mlm/model.safetensors
/kaggle/input/notebooks/aabdollahii/finetuning-on-wikifarsi/fabert_kg_mlm/checkpoint-12749/config.json
/kaggle/input/notebooks/aabdollahii/finetuning-on-wi

In [16]:
from pathlib import Path

INPUT_FILE = "/kaggle/input/notebooks/aabdollahii/persian-wiki-data-knowledge-graph-analysis/cleaned_knowledge_graph.csv"

TOKENIZER_PATH = "sbunlp/fabert"

OUTPUT_DIR = Path("/kaggle/working/factual_probe")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_FILE = OUTPUT_DIR / "factual_probe_dataset.csv"
REJECTED_FILE = OUTPUT_DIR / "factual_probe_rejected.csv"
STATS_FILE = OUTPUT_DIR / "factual_probe_stats.csv"

SAMPLES_PER_PREDICATE = 250

# This line of code does not let the system bias on a fact like iran
MAX_FACTS_PER_OBJECT_PER_PREDICATE = 30

RANDOM_SEED = 42


In [17]:
import ast
import html
import json
import random
import re
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
from transformers import AutoTokenizer

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)


In [18]:
def load_triples(path):
    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(f"Input file not found: {path}")

    suffix = path.suffix.lower()

    if suffix == ".csv":
        # sep=None attempts to detect comma, tab, semicolon, etc.
        return pd.read_csv(
            path,
            sep=None,
            engine="python",
            encoding="utf-8",
            on_bad_lines="skip",
        )

    if suffix == ".tsv":
        return pd.read_csv(
            path,
            sep="\t",
            encoding="utf-8",
            on_bad_lines="skip",
        )

    if suffix in {".jsonl", ".json"}:
        return pd.read_json(path, lines=suffix == ".jsonl")

    if suffix == ".parquet":
        return pd.read_parquet(path)

    raise ValueError(f"Unsupported input format: {suffix}")


raw_df = load_triples(INPUT_FILE)

print("Shape:", raw_df.shape)
print("Columns:", raw_df.columns.tolist())

display(raw_df.head())


Shape: (1848357, 7)
Columns: ['\ufeffSubject', 'Predicate', 'Object', 'subject_len', 'predicate_len', 'object_len', 'object_type']


,﻿Subject,Predicate,Object,subject_len,predicate_len,object_len,object_type
0,سعدی,نام,سعدی شیرازی,4,3,11,text
1,سعدی,زمینه فعالیت,شعر و نثر فارسی,4,12,15,text
2,سعدی,ملیت,ایرانی,4,4,6,text
3,سعدی,تاریخ تولد,بین ۵۸۵ تا ۶۱۵ هجری قمری,4,10,24,text
4,سعدی,محل تولد,شیراز، ایران,4,8,12,text


In [19]:
# reprocess some codes

COLUMN_ALIASES = {
    "subject": {
        "subject", "Subject", "SUBJECT",
        "subj", "entity", "head",
        "نهاد", "موضوع",
    },
    "predicate": {
        "predicate", "Predicate", "PREDICATE",
        "relation", "property", "rel",
        "رابطه", "ویژگی",
    },
    "object": {
        "object", "Object", "OBJECT",
        "obj", "value", "tail",
        "مفعول", "مقدار",
    },
}


def normalize_column_name(column):
    return str(column).strip().replace("\ufeff", "")


def find_column(df, possible_names):
    normalized_lookup = {
        normalize_column_name(column).lower(): column
        for column in df.columns
    }

    for name in possible_names:
        normalized_name = normalize_column_name(name).lower()
        if normalized_name in normalized_lookup:
            return normalized_lookup[normalized_name]

    return None


subject_col = find_column(raw_df, COLUMN_ALIASES["subject"])
predicate_col = find_column(raw_df, COLUMN_ALIASES["predicate"])
object_col = find_column(raw_df, COLUMN_ALIASES["object"])

if not all([subject_col, predicate_col, object_col]):
    raise ValueError(
        "Could not identify Subject, Predicate, and Object columns.\n"
        f"Available columns: {raw_df.columns.tolist()}\n"
        f"Detected: subject={subject_col}, "
        f"predicate={predicate_col}, object={object_col}"
    )

df = raw_df[[subject_col, predicate_col, object_col]].copy()

df.columns = ["subject", "predicate", "object"]

print("Detected columns:")
print({
    "subject": subject_col,
    "predicate": predicate_col,
    "object": object_col,
})

display(df.head())


Detected columns:
{'subject': '\ufeffSubject', 'predicate': 'Predicate', 'object': 'Object'}


,subject,predicate,object
0,سعدی,نام,سعدی شیرازی
1,سعدی,زمینه فعالیت,شعر و نثر فارسی
2,سعدی,ملیت,ایرانی
3,سعدی,تاریخ تولد,بین ۵۸۵ تا ۶۱۵ هجری قمری
4,سعدی,محل تولد,شیراز، ایران


In [20]:
MOJIBAKE_MARKERS = (
    "Ø",
    "Ù",
    "Û",
    "Ú",
    "Â",
    "Ã",
    "â€",
    "ð",
)


def mojibake_score(text):
    return sum(text.count(marker) for marker in MOJIBAKE_MARKERS)


def repair_mojibake_once(text):
    candidates = [text]

    for source_encoding in ("latin1", "cp1252"):
        try:
            repaired = text.encode(source_encoding).decode("utf-8")
            candidates.append(repaired)
        except (UnicodeEncodeError, UnicodeDecodeError):
            pass

    return min(candidates, key=mojibake_score)


def clean_text(value):
    if pd.isna(value):
        return ""

    text = str(value)
    text = html.unescape(text)

    # A maximum of two passes avoids accidental repeated conversions.
    for _ in range(2):
        repaired = repair_mojibake_once(text)
        if repaired == text:
            break
        text = repaired

    text = unicodedata.normalize("NFKC", text)

    replacements = {
        "\u200c": " ",  # Zero-width non-joiner
        "\u200d": "",
        "\ufeff": "",
        "\xa0": " ",
        "ي": "ی",
        "ى": "ی",
        "ك": "ک",
        "ۀ": "ه",
        "ة": "ه",
    }

    for old, new in replacements.items():
        text = text.replace(old, new)

    text = re.sub(r"\s+", " ", text).strip()
    return text


for column in ["subject", "predicate", "object"]:
    df[column] = df[column].map(clean_text)

display(df.head(20))


,subject,predicate,object
0,سعدی,نام,سعدی شیرازی
1,سعدی,زمینه فعالیت,شعر و نثر فارسی
2,سعدی,ملیت,ایرانی
3,سعدی,تاریخ تولد,بین ۵۸۵ تا ۶۱۵ هجری قمری
4,سعدی,محل تولد,شیراز، ایران
5,سعدی,تاریخ مرگ,بین ۶۹۰ تا ۶۹۵
6,سعدی,محل مرگ,شیراز، ایران
7,سعدی,محل زندگی,شیراز، بغداد
8,سعدی,مدفن,سعدیه، شیراز
9,سعدی,در زمان حکومت,اتابکان فارس، عباسیان، خوارزمشاهیان، ایلخانان


In [21]:
predicate_counts = (
    df["predicate"]
    .value_counts()
    .rename_axis("predicate")
    .reset_index(name="count")
)

print("Number of unique predicates:", df["predicate"].nunique())

display(predicate_counts.head(10))


Number of unique predicates: 14800


,predicate,count
0,استان,86362
1,شهرستان,82890
2,بخش,60290
3,جمعیت,55483
4,دهستان,54301
5,نام,24361
6,شماره ثبت,22984
7,ملیت,22332
8,کاربری,22034
9,زبان,20901


In [22]:
keywords = [
    "کشور",
    "استان",
    "ملیت",
    "زبان",
    "محل تولد",
    "زادگاه",
    "پایتخت",
]

pattern = "|".join(map(re.escape, keywords))

candidate_predicates = predicate_counts[
    predicate_counts["predicate"].str.contains(
        pattern,
        case=False,
        na=False,
    )
]

display(candidate_predicates)


,predicate,count
0,استان,86362
7,ملیت,22332
9,زبان,20901
14,کشور,17644
18,محل تولد,15493
...,...,...
14503,کشور تولید,1
14520,نوع_تقسیمات_کشوری۲,1
14521,نام_تقسیمات_کشوری۱,1
14522,نوع_تقسیمات_کشوری۱,1


In [23]:
# define templates

TEMPLATES = {
    "کشور": [
        {
            "template_id": "country_01",
            "template_type": "familiar",
            "text": "{subject} در کشور {mask} قرار دارد.",
        },
        {
            "template_id": "country_02",
            "template_type": "paraphrased",
            "text": "کشور محل قرارگیری {subject}، {mask} است.",
        },
        {
            "template_id": "country_03",
            "template_type": "paraphrased",
            "text": "{subject} متعلق به کشور {mask} است.",
        },
    ],
    "استان": [
        {
            "template_id": "province_01",
            "template_type": "familiar",
            "text": "{subject} در استان {mask} قرار دارد.",
        },
        {
            "template_id": "province_02",
            "template_type": "paraphrased",
            "text": "استان محل قرارگیری {subject}، {mask} است.",
        },
        {
            "template_id": "province_03",
            "template_type": "paraphrased",
            "text": "{subject} از مناطق استان {mask} است.",
        },
    ],
    "ملیت": [
        {
            "template_id": "nationality_01",
            "template_type": "familiar",
            "text": "ملیت {subject} {mask} است.",
        },
        {
            "template_id": "nationality_02",
            "template_type": "paraphrased",
            "text": "{subject} دارای ملیت {mask} است.",
        },
        {
            "template_id": "nationality_03",
            "template_type": "paraphrased",
            "text": "{subject} فردی {mask} است.",
        },
    ],
    "زبان": [
        {
            "template_id": "language_01",
            "template_type": "familiar",
            "text": "زبان {subject} {mask} است.",
        },
        {
            "template_id": "language_02",
            "template_type": "paraphrased",
            "text": "زبان مورد استفاده در {subject}، {mask} است.",
        },
        {
            "template_id": "language_03",
            "template_type": "paraphrased",
            "text": "{subject} با زبان {mask} مرتبط است.",
        },
    ],
    "زبان رسمی": [
        {
            "template_id": "official_language_01",
            "template_type": "familiar",
            "text": "زبان رسمی {subject} {mask} است.",
        },
        {
            "template_id": "official_language_02",
            "template_type": "paraphrased",
            "text": "مردم {subject} به‌طور رسمی به زبان {mask} صحبت می‌کنند.",
        },
        {
            "template_id": "official_language_03",
            "template_type": "paraphrased",
            "text": "در {subject}، {mask} زبان رسمی است.",
        },
    ],
    "محل تولد": [
        {
            "template_id": "birthplace_01",
            "template_type": "familiar",
            "text": "محل تولد {subject} {mask} است.",
        },
        {
            "template_id": "birthplace_02",
            "template_type": "paraphrased",
            "text": "{subject} در {mask} متولد شد.",
        },
        {
            "template_id": "birthplace_03",
            "template_type": "paraphrased",
            "text": "زادگاه {subject} {mask} است.",
        },
    ],
}


In [24]:
URL_PATTERN = re.compile(
    r"(https?://|www\.|\.com\b|\.org\b|\.ir\b)",
    flags=re.IGNORECASE,
)

PURE_NUMBER_PATTERN = re.compile(
    r"^[\d۰-۹٠-٩\s.,:/+\-–—٪%]+$"
)

INVALID_VALUES = {
    "",
    "nan",
    "none",
    "null",
    "نامشخص",
    "نامعلوم",
    "-",
    "_",
    "?",
}


def get_rejection_reason(row):
    subject = row["subject"]
    predicate = row["predicate"]
    object_value = row["object"]

    if subject.lower() in INVALID_VALUES:
        return "empty_subject"

    if predicate.lower() in INVALID_VALUES:
        return "empty_predicate"

    if object_value.lower() in INVALID_VALUES:
        return "empty_object"

    if URL_PATTERN.search(object_value):
        return "url_object"

    if PURE_NUMBER_PATTERN.fullmatch(object_value):
        return "numeric_object"

    if len(subject) > 200:
        return "long_subject"

    if len(object_value) > 100:
        return "long_object"

    if mojibake_score(subject) > 0:
        return "subject_encoding_problem"

    if mojibake_score(predicate) > 0:
        return "predicate_encoding_problem"

    if mojibake_score(object_value) > 0:
        return "object_encoding_problem"

    if predicate not in TEMPLATES:
        return "unsupported_predicate"

    return ""


initial_count = len(df)

df = df.drop_duplicates(
    subset=["subject", "predicate", "object"]
).reset_index(drop=True)

deduplicated_count = len(df)

df["rejection_reason"] = df.apply(get_rejection_reason, axis=1)

initial_rejected = df[df["rejection_reason"] != ""].copy()
candidates = df[df["rejection_reason"] == ""].copy()

print("Initial rows:", initial_count)
print("After deduplication:", deduplicated_count)
print("Supported candidates:", len(candidates))

display(
    candidates["predicate"]
    .value_counts()
    .rename_axis("predicate")
    .reset_index(name="count")
)


Initial rows: 1848357
After deduplication: 1823131
Supported candidates: 162778


,predicate,count
0,استان,86324
1,ملیت,22306
2,زبان,20889
3,کشور,17637
4,محل تولد,15479
5,زبان رسمی,143


In [25]:
tokenizer = AutoTokenizer.from_pretrained(
    TOKENIZER_PATH,
    use_fast=True,
)

print("Tokenizer:", tokenizer.name_or_path)
print("Mask token:", tokenizer.mask_token)
print("Mask token ID:", tokenizer.mask_token_id)


Tokenizer: sbunlp/fabert
Mask token: [MASK]
Mask token ID: 103


In [26]:
def tokenize_answer(answer):
    token_ids = tokenizer.encode(
        answer,
        add_special_tokens=False,
    )

    tokens = tokenizer.convert_ids_to_tokens(token_ids)

    return token_ids, tokens


def add_tokenization_info(row):
    token_ids, tokens = tokenize_answer(row["object"])

    return pd.Series({
        "object_token_count": len(token_ids),
        "gold_token_id": token_ids[0] if len(token_ids) == 1 else None,
        "object_tokens": json.dumps(tokens, ensure_ascii=False),
    })


token_info = candidates.apply(add_tokenization_info, axis=1)
candidates = pd.concat([candidates, token_info], axis=1)

display(
    candidates[
        [
            "subject",
            "predicate",
            "object",
            "object_token_count",
            "object_tokens",
        ]
    ].head(30)
)


,subject,predicate,object,object_token_count,object_tokens
2,سعدی,ملیت,ایرانی,1,"[""ایرانی""]"
4,سعدی,محل تولد,شیراز، ایران,3,"[""شیراز"", ""،"", ""ایران""]"
26,جامی,ملیت,ایرانی,1,"[""ایرانی""]"
43,عماد خراسانی,محل تولد,مشهد، ایران,3,"[""مشهد"", ""،"", ""ایران""]"
50,انتخابات مجلس شورای اسلامی (۱۳۸۳–۱۳۸۲),کشور,ایران,1,"[""ایران""]"
121,آنتوان دو سنت اگزوپری,ملیت,فرانسوی,1,"[""فرانسوی""]"
122,آنتوان دو سنت اگزوپری,محل تولد,لیون، فرانسه,3,"[""لیون"", ""،"", ""فرانسه""]"
140,اریک کلپتون,محل تولد,ریپلی، ساری، انگلستان,6,"[""ری"", ""##پلی"", ""،"", ""ساری"", ""،"", ""انگلستان""]"
195,ناصرخسرو,ملیت,ایرانی,1,"[""ایرانی""]"
197,ناصرخسرو,محل تولد,قبادیان، خراسان، غزنویان، تاجیکستان کنونی,9,"[""قبادی"", ""##ان"", ""،"", ""خراسان"", ""،"", ""غزنویان..."


In [27]:
single_token_df = candidates[
    candidates["object_token_count"] == 1
].copy()

multi_token_df = candidates[
    candidates["object_token_count"] != 1
].copy()

multi_token_df["rejection_reason"] = "multi_token_object"

print("Single-token facts:", len(single_token_df))
print("Multi-token facts:", len(multi_token_df))

display(
    single_token_df["predicate"]
    .value_counts()
    .rename_axis("predicate")
    .reset_index(name="single_token_count")
)


Single-token facts: 91889
Multi-token facts: 70889


,predicate,single_token_count
0,استان,42886
1,ملیت,17657
2,زبان,17269
3,کشور,11951
4,محل تولد,2080
5,زبان رسمی,46


In [29]:
def controlled_random_sample(
    dataframe,
    samples_per_predicate,
    max_per_object,
    random_seed,
):
    sampled_groups = []

    predicates = sorted(dataframe["predicate"].unique())

    for predicate_index, predicate in enumerate(predicates):
        predicate_df = dataframe[
            dataframe["predicate"] == predicate
        ].copy()

        # Randomize before limiting repeated objects.
        predicate_df = predicate_df.sample(
            frac=1,
            random_state=random_seed + predicate_index,
        )

        predicate_df = (
            predicate_df
            .groupby("object", group_keys=False)
            .head(max_per_object)
        )

        target_size = min(
            samples_per_predicate,
            len(predicate_df),
        )

        if target_size == 0:
            continue

        predicate_sample = predicate_df.sample(
            n=target_size,
            random_state=random_seed + 1000 + predicate_index,
        )

        sampled_groups.append(predicate_sample)

    if not sampled_groups:
        return pd.DataFrame(columns=dataframe.columns)

    result = pd.concat(
        sampled_groups,
        ignore_index=True,
    )

    return result.sample(
        frac=1,
        random_state=random_seed,
    ).reset_index(drop=True)


sampled_facts = controlled_random_sample(
    dataframe=single_token_df,
    samples_per_predicate=SAMPLES_PER_PREDICATE,
    max_per_object=MAX_FACTS_PER_OBJECT_PER_PREDICATE,
    random_seed=RANDOM_SEED,
)

sampled_facts.insert(
    0,
    "fact_id",
    [f"fact_{index:06d}" for index in range(1, len(sampled_facts) + 1)],
)

sampled_facts["split_type"] = "seen"
sampled_facts["source"] = str(INPUT_FILE)

print("Selected facts:", len(sampled_facts))

display(
    sampled_facts["predicate"]
    .value_counts()
    .rename_axis("predicate")
    .reset_index(name="selected_count")
)


Selected facts: 1296


,predicate,selected_count
0,محل تولد,250
1,زبان,250
2,ملیت,250
3,کشور,250
4,استان,250
5,زبان رسمی,46


In [30]:
def choose_template(predicate, rng):
    predicate_templates = TEMPLATES[predicate]
    return rng.choice(predicate_templates)


rng = random.Random(RANDOM_SEED)

dataset_rows = []

for row in sampled_facts.itertuples(index=False):
    template = choose_template(row.predicate, rng)

    prompt = template["text"].format(
        subject=row.subject,
        mask=tokenizer.mask_token,
    )

    dataset_rows.append({
        "fact_id": row.fact_id,
        "subject": row.subject,
        "predicate": row.predicate,
        "object": row.object,
        "prompt": prompt,
        "template_id": template["template_id"],
        "template_type": template["template_type"],
        "gold_answer": row.object,
        "gold_token_id": int(row.gold_token_id),
        "object_token_count": int(row.object_token_count),
        "object_tokens": row.object_tokens,
        "split_type": row.split_type,
        "source": row.source,
        "valid": True,
    })


probe_df = pd.DataFrame(dataset_rows)

print("Probe rows:", len(probe_df))
display(probe_df.head(20))


Probe rows: 1296


,fact_id,subject,predicate,object,prompt,template_id,template_type,gold_answer,gold_token_id,object_token_count,object_tokens,split_type,source,valid
0,fact_000001,محمد پورستار,محل تولد,اردبیل,زادگاه محمد پورستار [MASK] است.,birthplace_03,paraphrased,اردبیل,10050,1,"[""اردبیل""]",seen,/kaggle/input/notebooks/aabdollahii/persian-wi...,True
1,fact_000002,بخش ایرندگان,زبان,بلوچی,زبان بخش ایرندگان [MASK] است.,language_01,familiar,بلوچی,31449,1,"[""بلوچی""]",seen,/kaggle/input/notebooks/aabdollahii/persian-wi...,True
2,fact_000003,پرم چوپرا,محل تولد,لاهور,محل تولد پرم چوپرا [MASK] است.,birthplace_01,familiar,لاهور,49461,1,"[""لاهور""]",seen,/kaggle/input/notebooks/aabdollahii/persian-wi...,True
3,fact_000004,یاروسلاو سایفرت,ملیت,چکی,یاروسلاو سایفرت فردی [MASK] است.,nationality_03,paraphrased,چکی,36644,1,"[""چکی""]",seen,/kaggle/input/notebooks/aabdollahii/persian-wi...,True
4,fact_000005,سه برخوانی,کشور,تهران,کشور محل قرارگیری سه برخوانی، [MASK] است.,country_02,paraphrased,تهران,3148,1,"[""تهران""]",seen,/kaggle/input/notebooks/aabdollahii/persian-wi...,True
5,fact_000006,پاستورا سولر,ملیت,اسپانیایی,ملیت پاستورا سولر [MASK] است.,nationality_01,familiar,اسپانیایی,17013,1,"[""اسپانیایی""]",seen,/kaggle/input/notebooks/aabdollahii/persian-wi...,True
6,fact_000007,اسکامپولو ۵۳ (فیلم ۱۹۵۳),زبان,ایتالیایی,زبان اسکامپولو ۵۳ (فیلم ۱۹۵۳) [MASK] است.,language_01,familiar,ایتالیایی,13857,1,"[""ایتالیایی""]",seen,/kaggle/input/notebooks/aabdollahii/persian-wi...,True
7,fact_000008,قباد آذرآیین,محل تولد,مسجدسلیمان,محل تولد قباد آذرآیین [MASK] است.,birthplace_01,familiar,مسجدسلیمان,32192,1,"[""مسجدسلیمان""]",seen,/kaggle/input/notebooks/aabdollahii/persian-wi...,True
8,fact_000009,ژرژ لامپن,ملیت,فرانسه,ژرژ لامپن فردی [MASK] است.,nationality_03,paraphrased,فرانسه,5916,1,"[""فرانسه""]",seen,/kaggle/input/notebooks/aabdollahii/persian-wi...,True
9,fact_000010,کشکش,استان,گیلان,کشکش در استان [MASK] قرار دارد.,province_01,familiar,گیلان,8188,1,"[""گیلان""]",seen,/kaggle/input/notebooks/aabdollahii/persian-wi...,True


In [31]:
def validate_probe_row(row):
    errors = []

    if row["prompt"].count(tokenizer.mask_token) != 1:
        errors.append("mask_count_not_one")

    if row["object"] in row["prompt"]:
        errors.append("gold_answer_leaked")

    if not row["subject"]:
        errors.append("empty_subject")

    if not row["predicate"]:
        errors.append("empty_predicate")

    if row["object_token_count"] != 1:
        errors.append("not_single_token")

    return "|".join(errors)


probe_df["validation_error"] = probe_df.apply(
    validate_probe_row,
    axis=1,
)

probe_df["valid"] = probe_df["validation_error"].eq("")

invalid_probe_rows = probe_df[~probe_df["valid"]].copy()
probe_df = probe_df[probe_df["valid"]].copy()

print("Valid probes:", len(probe_df))
print("Invalid probes:", len(invalid_probe_rows))

if len(invalid_probe_rows):
    display(invalid_probe_rows.head(20))


Valid probes: 1206
Invalid probes: 90


,fact_id,subject,predicate,object,prompt,template_id,template_type,gold_answer,gold_token_id,object_token_count,object_tokens,split_type,source,valid,validation_error
17,fact_000018,طایر اردبیلی,محل تولد,اردبیل,محل تولد طایر اردبیلی [MASK] است.,birthplace_01,familiar,اردبیل,10050,1,"[""اردبیل""]",seen,/kaggle/input/notebooks/aabdollahii/persian-wi...,False,gold_answer_leaked
19,fact_000020,ویکی پدیای کاتالان,زبان,کاتالان,زبان ویکی پدیای کاتالان [MASK] است.,language_01,familiar,کاتالان,45249,1,"[""کاتالان""]",seen,/kaggle/input/notebooks/aabdollahii/persian-wi...,False,gold_answer_leaked
43,fact_000044,فردی اردبیلی,محل تولد,اردبیل,فردی اردبیلی در [MASK] متولد شد.,birthplace_02,paraphrased,اردبیل,10050,1,"[""اردبیل""]",seen,/kaggle/input/notebooks/aabdollahii/persian-wi...,False,gold_answer_leaked
46,fact_000047,فهرست میراث جهانی در تونس,کشور,تونس,کشور محل قرارگیری فهرست میراث جهانی در تونس، [...,country_02,paraphrased,تونس,18527,1,"[""تونس""]",seen,/kaggle/input/notebooks/aabdollahii/persian-wi...,False,gold_answer_leaked
82,fact_000083,مارو (خواننده پرتغالی),ملیت,پرتغالی,مارو (خواننده پرتغالی) دارای ملیت [MASK] است.,nationality_02,paraphrased,پرتغالی,23011,1,"[""پرتغالی""]",seen,/kaggle/input/notebooks/aabdollahii/persian-wi...,False,gold_answer_leaked
87,fact_000088,بیضای اردبیلی,محل تولد,اردبیل,زادگاه بیضای اردبیلی [MASK] است.,birthplace_03,paraphrased,اردبیل,10050,1,"[""اردبیل""]",seen,/kaggle/input/notebooks/aabdollahii/persian-wi...,False,gold_answer_leaked
91,fact_000092,فهرست میراث جهانی در قبرس,کشور,قبرس,فهرست میراث جهانی در قبرس متعلق به کشور [MASK]...,country_03,paraphrased,قبرس,30892,1,"[""قبرس""]",seen,/kaggle/input/notebooks/aabdollahii/persian-wi...,False,gold_answer_leaked
95,fact_000096,امیرآباد (اردبیل),استان,اردبیل,امیرآباد (اردبیل) در استان [MASK] قرار دارد.,province_01,familiar,اردبیل,10050,1,"[""اردبیل""]",seen,/kaggle/input/notebooks/aabdollahii/persian-wi...,False,gold_answer_leaked
97,fact_000098,سیدآباد (قم),استان,قم,استان محل قرارگیری سیدآباد (قم)، [MASK] است.,province_02,paraphrased,قم,4609,1,"[""قم""]",seen,/kaggle/input/notebooks/aabdollahii/persian-wi...,False,gold_answer_leaked
181,fact_000182,جام ولیعهد قطر ۲۰۰۹,کشور,قطر,کشور محل قرارگیری جام ولیعهد قطر ۲۰۰۹، [MASK] ...,country_02,paraphrased,قطر,6573,1,"[""قطر""]",seen,/kaggle/input/notebooks/aabdollahii/persian-wi...,False,gold_answer_leaked


In [32]:
def build_stats(
    full_df,
    candidates_df,
    single_df,
    sampled_df,
    final_df,
):
    predicates = sorted(
        set(full_df["predicate"])
        | set(TEMPLATES.keys())
    )

    stats_rows = []

    for predicate in predicates:
        raw_predicate = full_df[
            full_df["predicate"] == predicate
        ]

        candidate_predicate = candidates_df[
            candidates_df["predicate"] == predicate
        ]

        single_predicate = single_df[
            single_df["predicate"] == predicate
        ]

        sampled_predicate = sampled_df[
            sampled_df["predicate"] == predicate
        ]

        final_predicate = final_df[
            final_df["predicate"] == predicate
        ]

        if len(raw_predicate) == 0 and predicate not in TEMPLATES:
            continue

        most_common_object = ""
        most_common_object_count = 0
        most_common_object_share = 0.0

        if len(final_predicate):
            object_counts = final_predicate["object"].value_counts()

            most_common_object = object_counts.index[0]
            most_common_object_count = int(object_counts.iloc[0])
            most_common_object_share = (
                most_common_object_count / len(final_predicate)
            )

        stats_rows.append({
            "predicate": predicate,
            "raw_count": len(raw_predicate),
            "clean_candidate_count": len(candidate_predicate),
            "single_token_count": len(single_predicate),
            "multi_token_count": (
                len(candidate_predicate) - len(single_predicate)
            ),
            "sampled_fact_count": len(sampled_predicate),
            "final_probe_count": len(final_predicate),
            "unique_objects": final_predicate["object"].nunique(),
            "most_common_object": most_common_object,
            "most_common_object_count": most_common_object_count,
            "most_common_object_share": most_common_object_share,
        })

    return pd.DataFrame(stats_rows)


stats_df = build_stats(
    full_df=df,
    candidates_df=candidates,
    single_df=single_token_df,
    sampled_df=sampled_facts,
    final_df=probe_df,
)

display(stats_df)


,predicate,raw_count,clean_candidate_count,single_token_count,multi_token_count,sampled_fact_count,final_probe_count,unique_objects,most_common_object,most_common_object_count,most_common_object_share
0,(58بخش)num_episodes,1,0,0,0,0,0,0,,0,0.0
1,(IATA,1,0,0,0,0,0,0,,0,0.0
2,(unranked),1,0,0,0,0,0,0,,0,0.0
3,(نام محلی,1,0,0,0,0,0,0,,0,0.0
4,- ! bgcolor,1,0,0,0,0,0,0,,0,0.0
...,...,...,...,...,...,...,...,...,...,...,...
14795,۴داده۵,1,0,0,0,0,0,0,,0,0.0
14796,۴داده۶,1,0,0,0,0,0,0,,0,0.0
14797,۴داده۷,1,0,0,0,0,0,0,,0,0.0
14798,۷۰(تاکنون),1,0,0,0,0,0,0,,0,0.0


In [33]:
final_columns = [
    "fact_id",
    "subject",
    "predicate",
    "object",
    "prompt",
    "template_id",
    "template_type",
    "gold_answer",
    "gold_token_id",
    "object_token_count",
    "object_tokens",
    "split_type",
    "source",
    "valid",
    "validation_error",
]

probe_df = probe_df[final_columns].reset_index(drop=True)

all_rejected = pd.concat(
    [
        initial_rejected,
        multi_token_df,
        invalid_probe_rows,
    ],
    ignore_index=True,
    sort=False,
)

probe_df.to_csv(
    OUTPUT_FILE,
    index=False,
    encoding="utf-8-sig",
)

all_rejected.to_csv(
    REJECTED_FILE,
    index=False,
    encoding="utf-8-sig",
)

stats_df.to_csv(
    STATS_FILE,
    index=False,
    encoding="utf-8-sig",
)

print("Saved files:")
print(OUTPUT_FILE)
print(REJECTED_FILE)
print(STATS_FILE)


Saved files:
/kaggle/working/factual_probe/factual_probe_dataset.csv
/kaggle/working/factual_probe/factual_probe_rejected.csv
/kaggle/working/factual_probe/factual_probe_stats.csv


In [34]:
print("Final dataset shape:", probe_df.shape)
print()

print("Facts by predicate:")
print(probe_df["predicate"].value_counts())
print()

print("Templates:")
print(probe_df["template_id"].value_counts())
print()

print("Split types:")
print(probe_df["split_type"].value_counts())
print()

display(probe_df.sample(
    n=min(30, len(probe_df)),
    random_state=RANDOM_SEED,
))


Final dataset shape: (1206, 15)

Facts by predicate:
predicate
ملیت         242
زبان         240
محل تولد     238
استان        233
کشور         208
زبان رسمی     45
Name: count, dtype: int64

Templates:
template_id
province_02             88
nationality_01          85
nationality_03          84
birthplace_02           83
language_03             81
language_01             81
birthplace_03           78
language_02             78
birthplace_01           77
province_03             77
nationality_02          73
country_01              72
country_03              72
province_01             68
country_02              64
official_language_02    17
official_language_03    17
official_language_01    11
Name: count, dtype: int64

Split types:
split_type
seen    1206
Name: count, dtype: int64



,fact_id,subject,predicate,object,prompt,template_id,template_type,gold_answer,gold_token_id,object_token_count,object_tokens,split_type,source,valid,validation_error
101,fact_000111,بی نهر علیا,استان,کرمانشاه,استان محل قرارگیری بی نهر علیا، [MASK] است.,province_02,paraphrased,کرمانشاه,8746,1,"[""کرمانشاه""]",seen,/kaggle/input/notebooks/aabdollahii/persian-wi...,True,
260,fact_000280,آقای نابغه (فیلم),کشور,هند,آقای نابغه (فیلم) در کشور [MASK] قرار دارد.,country_01,familiar,هند,4837,1,"[""هند""]",seen,/kaggle/input/notebooks/aabdollahii/persian-wi...,True,
777,fact_000836,آکادمی بررا,کشور,ایتالیا,آکادمی بررا متعلق به کشور [MASK] است.,country_03,paraphrased,ایتالیا,8564,1,"[""ایتالیا""]",seen,/kaggle/input/notebooks/aabdollahii/persian-wi...,True,
109,fact_000119,محمود پیراسته,محل تولد,بوشهر,زادگاه محمود پیراسته [MASK] است.,birthplace_03,paraphrased,بوشهر,9191,1,"[""بوشهر""]",seen,/kaggle/input/notebooks/aabdollahii/persian-wi...,True,
649,fact_000699,خرگی,استان,هرمزگان,استان محل قرارگیری خرگی، [MASK] است.,province_02,paraphrased,هرمزگان,14931,1,"[""هرمزگان""]",seen,/kaggle/input/notebooks/aabdollahii/persian-wi...,True,
735,fact_000789,فرانسواز راشمول,محل تولد,نانسی,زادگاه فرانسواز راشمول [MASK] است.,birthplace_03,paraphrased,نانسی,39373,1,"[""نانسی""]",seen,/kaggle/input/notebooks/aabdollahii/persian-wi...,True,
332,fact_000355,آوین سفلی,استان,هرمزگان,آوین سفلی از مناطق استان [MASK] است.,province_03,paraphrased,هرمزگان,14931,1,"[""هرمزگان""]",seen,/kaggle/input/notebooks/aabdollahii/persian-wi...,True,
49,fact_000054,اخطار (فیلم ۲۰۱۸),کشور,اسپانیا,اخطار (فیلم ۲۰۱۸) متعلق به کشور [MASK] است.,country_03,paraphrased,اسپانیا,9563,1,"[""اسپانیا""]",seen,/kaggle/input/notebooks/aabdollahii/persian-wi...,True,
461,fact_000494,علی عباس قدیروف,محل تولد,باکو,علی عباس قدیروف در [MASK] متولد شد.,birthplace_02,paraphrased,باکو,19635,1,"[""باکو""]",seen,/kaggle/input/notebooks/aabdollahii/persian-wi...,True,
980,fact_001051,امامزاده بزم,استان,فارس,امامزاده بزم از مناطق استان [MASK] است.,province_03,paraphrased,فارس,4872,1,"[""فارس""]",seen,/kaggle/input/notebooks/aabdollahii/persian-wi...,True,
